### Carga de librerías

In [1]:
import os
import numpy as np
import pandas as pd
from math import pi
import seaborn as sns
import No_Supervisados as ns
import matplotlib.pyplot as plt
from prince import PCA as PCA_Prince
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, ward, single, complete, average, linkage, fcluster
from Sem3_MD_Funciones import ACP
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'numpy'

In [ ]:
pd.options.display.max_rows = 10

## 1. Descripción del set de datos y tipos de variables

In [ ]:
datos = pd.read_csv('Wholesale customers data.csv', delimiter=',', decimal=".")
datos

In [ ]:
datos.shape

In [ ]:
datos.dtypes

**Descripción del set de datos:** el Wholesale Customers Dataset (repositorio UCI) contiene el gasto anual, en unidades monetarias, de **440 clientes** de un distribuidor mayorista en Portugal, en distintas categorías de productos. No posee una variable objetivo, por lo que se trata de un **aprendizaje no supervisado** con el objetivo de encontrar grupos (segmentos) de clientes con patrones de compra similares.

| Variable | Tipo | Descripción | Uso en el análisis |
|---|---|---|---|
| Channel | Categórica nominal (codificada 1-2) | Canal: 1 = Horeca (hoteles, restaurantes, cafés), 2 = Retail (minoristas) | Descriptiva: interpretar los clústeres |
| Region | Categórica nominal (codificada 1-3) | Región: 1 = Lisboa, 2 = Oporto, 3 = Otra | Descriptiva: interpretar los clústeres |
| Fresh | Numérica continua | Gasto anual en productos frescos | Variable de clusterización |
| Milk | Numérica continua | Gasto anual en lácteos | Variable de clusterización |
| Grocery | Numérica continua | Gasto anual en abarrotes | Variable de clusterización |
| Frozen | Numérica continua | Gasto anual en congelados | Variable de clusterización |
| Detergents_Paper | Numérica continua | Gasto anual en detergentes y productos de papel | Variable de clusterización |
| Delicassen | Numérica continua | Gasto anual en productos delicatessen | Variable de clusterización |

Aunque Python lee `Channel` y `Region` como `int64`, en realidad son **categóricas**: sus números son solo códigos y no tiene sentido calcular distancias o promedios sobre ellos. Por eso se recodifican con la función `recodificar` vista en clase.

### Recodificación de variables categóricas

In [ ]:
datos["Channel"] = ns.recodificar(datos["Channel"], {1:'Horeca', 2:'Retail'})
datos["Region"] = ns.recodificar(datos["Region"], {1:'Lisboa', 2:'Oporto', 3:'Otra'})
datos.head()

In [ ]:
datos.dtypes

## 2. Análisis exploratorio
### Valores nulos y duplicados

In [ ]:
print("Valores nulos por variable:\n", datos.isnull().sum())
print("\nRegistros duplicados:", datos.duplicated().sum())

El set de datos **no presenta valores nulos ni registros duplicados**, por lo que no es necesario imputar ni eliminar observaciones.

### Estadísticas descriptivas de las variables numéricas

In [ ]:
datos_numericos = datos.select_dtypes(include=["number"])
datos_numericos.describe().round(2)

- Todas las variables presentan una **media bastante mayor que la mediana** (por ejemplo `Fresh`: media 12 000 vs mediana 8 504; `Detergents_Paper`: 2 881 vs 816), lo cual indica distribuciones **asimétricas a la derecha**: la mayoría de clientes compra poco y unos pocos compran montos muy altos.
- Los máximos son muy superiores al percentil 75 (`Fresh` llega a 112 151 y `Delicassen` a 47 943), lo que anticipa la presencia de **valores atípicos**.
- Las escalas son muy distintas: `Fresh` tiene una desviación estándar de ~12 600, mientras que `Delicassen` ~2 800.

### Frecuencias de las variables categóricas

In [ ]:
print(datos["Channel"].value_counts())
print()
print(datos["Region"].value_counts())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (15,4), dpi = 100)
sns.countplot(x = "Channel", data = datos, ax = ax[0])
ax[0].set_title("Clientes por Canal")
sns.countplot(x = "Region", data = datos, ax = ax[1])
ax[1].set_title("Clientes por Región")
sns.countplot(x = "Region", hue = "Channel", data = datos, ax = ax[2])
ax[2].set_title("Región vs Canal")
plt.show()

- El canal **Horeca** concentra la mayoría de clientes (298, 68 %) frente a **Retail** (142, 32 %).
- La región **"Otra"** agrupa 316 clientes (72 %); Lisboa tiene 77 y Oporto 47.
- La proporción de clientes Horeca predomina en las tres regiones
- y Oporto tiene proporcionalmente más clientes Retail.

### Distribución de las variables numéricas (histogramas)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize = (15,8), dpi = 100)
for i, col in enumerate(datos_numericos.columns):
    sns.histplot(datos_numericos[col], kde = True, ax = ax[i//3, i%3])
    ax[i//3, i%3].set_title(col)
plt.tight_layout()
plt.show()

Los histogramas muestran una fuerte asimetría positiva en todas las categorías de gasto, más marcada en Frozen, Detergents_Paper y Delicassen. La mayoría de los clientes gasta montos bajos y unos pocos concentran gastos muy altos, lo que sugiere outliers o subpoblaciones.


### Diagramas de caja (valores atípicos)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (12,6), dpi = 100)
sns.boxplot(data = datos_numericos, ax = ax)
ax.set_title("Diagramas de caja del gasto anual por categoría")
plt.show()

In [ ]:
# Cantidad de valores atípicos por variable (criterio del rango intercuartílico)
Q1 = datos_numericos.quantile(0.25)
Q3 = datos_numericos.quantile(0.75)
RIC = Q3 - Q1
((datos_numericos < (Q1 - 1.5 * RIC)) | (datos_numericos > (Q3 + 1.5 * RIC))).sum()

Todas las variables presentan **valores atípicos** (entre 20 y 43 por variable según el criterio de 1.5 × RIC); `Frozen` es la que tiene más (43) y `Delicassen` tiene el caso más extremo. Estos clientes no son errores de captura sino **clientes grandes reales**, por lo que se mantienen en el análisis.

### Gasto por canal

In [ ]:
fig, ax = plt.subplots(2, 3, figsize = (15,8), dpi = 100)
for i, col in enumerate(datos_numericos.columns):
    sns.boxplot(x = "Channel", y = col, data = datos, ax = ax[i//3, i%3])
    ax[i//3, i%3].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
datos.groupby("Channel")[datos_numericos.columns].mean().round(2)

Existe una diferencia clara en el patrón de compra según el canal:
- **Horeca** gasta más en `Fresh` (13 476) y `Frozen` (3 748), algo coherente con restaurantes y hoteles que preparan comida.
- **Retail** gasta mucho más en `Grocery` (16 323 vs 3 962), `Milk` (10 717 vs 3 452) y `Detergents_Paper` (7 270 vs 791), que son productos de reventa típicos de un supermercado.

Esto sugiere que los clústeres podrían reflejar en buena medida el canal del cliente.

### Matriz de correlación

In [ ]:
corr = datos_numericos.corr()
fig, ax = plt.subplots(1, 1, figsize = (9,7), dpi = 100)
sns.heatmap(corr, annot = True, fmt = ".2f", cmap = "coolwarm", vmin = -1, vmax = 1, ax = ax)
ax.set_title("Matriz de correlación")
plt.show()

- **Grocery y Detergents_Paper** tienen una correlación muy alta (**0.92**), y ambas se correlacionan fuertemente con **Milk** (0.73 y 0.66). Estas tres variables forman un bloque asociado a las compras tipo *Retail*.
- **Fresh, Frozen y Delicassen** tienen correlaciones positivas moderadas entre sí (0.24 a 0.39), formando un segundo bloque asociado a *Horeca*.
- Entre ambos bloques las correlaciones son cercanas a 0.

### Estandarización de los datos

**¿Por qué estandarizar?** Las variables de gasto están en la misma unidad monetaria, pero sus escalas y varianzas son muy diferentes (por ejemplo, `Fresh` tiene una desviación estándar mucho mayor que `Delicassen`). Tanto K-medias como la clusterización jerárquica utilizan la **distancia euclidiana**, por lo que, sin estandarizar, las variables con mayor varianza dominarían el cálculo de las distancias y los grupos se formarían prácticamente solo por ellas. Con `StandardScaler` cada variable queda con media 0 y desviación estándar 1, de modo que todas aportan de forma equitativa al análisis.

Se decidió **no eliminar los valores atípicos**, ya que corresponden a clientes reales con compras muy grandes (información valiosa para el negocio); sin embargo, se debe tener presente que pueden formar grupos pequeños propios.

Las variables categóricas `Channel` y `Region` **no se utilizan para formar los clústeres**: se reservan para describir e interpretar los grupos obtenidos.

In [ ]:
scaler = StandardScaler()
datos_std = pd.DataFrame(scaler.fit_transform(datos_numericos), columns = datos_numericos.columns)
datos_std.describe().round(2)

### Análisis en Componentes Principales (ACP)

In [ ]:
acp = ACP(datos_std, n_componentes = 6)
acp.var_explicada

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (8,5), dpi = 100)
plt.bar(range(len(acp.var_explicada)), acp.var_explicada)
plt.plot(range(len(acp.var_explicada)), np.cumsum(acp.var_explicada), 'o-', c = 'k')
plt.xlabel("Componente")
plt.ylabel("% de varianza explicada")
plt.title("Varianza explicada por componente")
plt.show()

El gráfico de varianza explicada muestra que la información de las seis variables de gasto se concentra en pocos componentes principales. El primer componente explica cerca del 44% de la varianza y el segundo cerca del 28%, de modo que entre ambos acumulan aproximadamente el 72%. Con tres componentes se alcanza cerca del 85%, y con cuatro cerca del 94%, mientras que el quinto y el sexto aportan muy poco. La curva acumulada se aplana a partir del tercer componente, lo que indica un "codo" y sugiere que es posible reducir la dimensionalidad de seis a dos o tres componentes sin perder gran parte de la información. Esto confirma que existe redundancia entre las categorías de gasto, y que retener tres componentes.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (12,8), dpi = 100)
acp.plot_plano_principal(ind_labels = False)
plt.show()

El plano principal muestra la proyección de los clientes sobre los dos primeros componentes, que en conjunto explican cerca del 72% de la varianza. La mayoría de los clientes se agrupa en una nube densa cerca del origen, lo que corresponde a patrones de gasto bajos o moderados, mientras que una cola se extiende hacia valores altos del primer componente. Además, se aprecian algunos puntos muy alejados del resto, que pueden corresponder a clientes con gasto excepcionalmente alto y que se consideran posibles valores atípicos. No se observan grupos claramente separados, lo que sugiere una distribución continua de los clientes.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (8,8), dpi = 100)
acp.plot_circulo()
plt.show()

- Las dos primeras componentes explican **72.46 %** de la varianza (44.08 % + 28.38 %), por lo que el plano principal representa bien a los datos.
- En el círculo de correlación se observan los dos bloques identificados en la matriz de correlación: **Grocery, Detergents_Paper y Milk** se asocian principalmente con la **Componente 0** (eje de compras tipo *Retail*), mientras que **Fresh, Frozen y Delicassen** se asocian con la **Componente 1** (eje de compras tipo *Horeca*).
- En el plano principal la mayoría de clientes se concentra cerca del origen y unos pocos se alejan mucho; estos son los clientes atípicos de gran volumen.

## 3. Clusterización K-medias
### Método del codo para escoger K

In [ ]:
inercias = []
for k in range(1, 11):
    kmedias = KMeans(n_clusters = k, n_init = 10, random_state = 42)
    kmedias.fit(datos_std)
    inercias.append(kmedias.inertia_)

fig, ax = plt.subplots(1, 1, figsize = (8,5), dpi = 100)
plt.plot(range(1, 11), inercias, 'o-')
plt.xlabel("Número de clústeres (K)")
plt.ylabel("Inercia intra-clases")
plt.title("Método del codo")
plt.show()

La inercia cae fuertemente de K = 1 a K = 2 y luego disminuye de forma más gradual, sin un codo muy marcado (algo común cuando hay muchos valores atípicos). Los valores candidatos están aproximadamente entre **K = 3 y K = 5**, por lo que la decisión se complementa revisando el tamaño de los grupos.

### Tamaño de los grupos según K
Además del codo, se revisa cuántos clientes quedan en cada grupo para distintos valores de K, ya que por los valores atípicos algunos valores de K generan grupos de 1 o 2 clientes que no son útiles para el negocio.

In [ ]:
for k in range(2, 8):
    kmedias = KMeans(n_clusters = k, n_init = 10, random_state = 42)
    kmedias.fit(datos_std)
    print("K =", k, "->", sorted(pd.Series(kmedias.labels_).value_counts().tolist(), reverse = True))

- Con **K = 2 y K = 3** los atípicos dominan: K = 3 produce un grupo de solo **2 clientes**, y el grupo principal tiene casi 400 clientes, lo que aporta poca segmentación.
- Con **K = 4** se obtienen dos grupos grandes y bien diferenciados (315 y 110 clientes) más dos grupos pequeños (10 y 5) que aíslan a los clientes de gran volumen.
- Con K ≥ 5 empiezan a aparecer grupos de 1 o 2 clientes.

Por lo tanto, se escoge **K = 4**.

In [ ]:
kmedias = KMeans(n_clusters = 4, n_init = 10, random_state = 42)
kmedias.fit(datos_std)
grupos_km = kmedias.predict(datos_std)
grupos_km

In [ ]:
pd.Series(grupos_km).value_counts().sort_index()

### Centros de los clústeres (datos estandarizados)

In [ ]:
centros = np.array(pd.concat([ns.centroide(0, datos_std, grupos_km),
                              ns.centroide(1, datos_std, grupos_km),
                              ns.centroide(2, datos_std, grupos_km),
                              ns.centroide(3, datos_std, grupos_km)]))
centros

In [ ]:
ns.bar_plot(centros, datos_std.columns)
plt.show()

El gráfico de centroides muestra el perfil de gasto promedio de cada uno de los cuatro clusters, expresado en unidades estandarizadas. El cluster 0 presenta un gasto moderado con una ligera inclinación hacia Milk, Grocery y Detergents_Paper y el cluster 1 agrupa clientes con gasto bajo en todas las categorías, El cluster 2 se caracteriza por un gasto muy elevado en Detergents_Paper, Grocery y Milk, correspondiente a un perfil de tipo retail, mientras que el cluster 3 destaca por un gasto muy alto en Delicassen, Frozen y Fresh, asociado a productos frescos y de servicio alimentario.

In [ ]:
ns.radar_plot(centros, datos_std.columns)
plt.show()

### Promedios en la escala original y composición por canal y región

In [ ]:
datos["Cluster_KM"] = grupos_km
datos.groupby("Cluster_KM")[datos_numericos.columns].mean().round(2)

In [ ]:
pd.crosstab(datos["Cluster_KM"], datos["Channel"])

In [ ]:
pd.crosstab(datos["Cluster_KM"], datos["Region"])

### Clústeres sobre el plano principal del ACP

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (10,7), dpi = 100)
coord = acp.coordenadas_ind
sns.scatterplot(x = coord[0], y = coord[1], hue = grupos_km, palette = "tab10", ax = ax)
ax.set_xlabel("Componente 0")
ax.set_ylabel("Componente 1")
ax.set_title("Clústeres K-medias en el plano principal")
plt.show()

**Interpretación de los clústeres K-medias (K = 4):**

- **Cluster 0 Minoristas (Retail) típicos (110 clientes):** gasto alto en Grocery (15 600), Milk (10 100) y Detergents_Paper (6 700), y bajo en Fresh y Frozen. El 85 % pertenece al canal Retail.
- **Cluster 1 Horeca típicos (315 clientes):** es el grupo más grande. Su gasto se concentra en Fresh (13 500) y Frozen (3 300), con montos bajos en abarrotes, lácteos y detergentes. El 88 % pertenece al canal Horeca.
- **Cluster 2 Grandes minoristas (10 clientes):** todos del canal Retail. Tienen montos muy altos en Grocery (48 500), Milk (34 700) y Detergents_Paper (24 900); podrían ser supermercados o cadenas grandes.
- **Cluster 3
Grandes compradores de frescos y delicatessen (5 clientes):** gasto extremadamente alto en Fresh (50 300), Frozen (25 100) y Delicassen (18 600), y también alto en Milk (26 400), pero bajo en Detergents_Paper; 4 de 5 son Horeca (por ejemplo, hoteles o cadenas de restaurantes grandes).

La **región no diferencia** a los grupos: en todos los clústeres predomina "Otra" en una proporción similar a la del total. En el plano principal se ve que el Cluster 0 y el 2 se extienden hacia el eje de *Retail*, mientras que el 3 lo hace hacia el eje de *Horeca*.

## 4. Clusterización jerárquica (comparación)

In [ ]:
ward_res = ward(datos_std)          #Ward
average_res = average(datos_std)    #Promedio
single_res  = single(datos_std)     #Salto mínimo
complete_res = complete(datos_std)  #Salto Máximo

### Salto mínimo, salto máximo y promedio

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (18,5), dpi = 100)
dendrogram(single_res, no_labels = True, ax = ax[0])
ax[0].set_title("Salto mínimo")
dendrogram(complete_res, no_labels = True, ax = ax[1])
ax[1].set_title("Salto máximo")
dendrogram(average_res, no_labels = True, ax = ax[2])
ax[2].set_title("Promedio")
plt.show()

Con **salto mínimo** se forma un efecto de "cadena" (los clientes se van uniendo uno a uno) y con **promedio** pasa algo similar: ambos solo separan a los clientes atípicos. El **salto máximo** forma grupos algo más balanceados. Por eso se utiliza el método de **Ward**, que minimiza la varianza dentro de los grupos y es el más parecido a K-medias.

### Ward

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (15,8), dpi = 200)
dendrogram(ward_res, no_labels = True, ax = ax)

# Agrega cortes con 2 y 3 clústeres con agregación de Ward
limites = ax.get_xbound()
ax.plot(limites, [30, 30], '--', c='k')
ax.plot(limites, [26, 26], '--', c='k')
ax.text(limites[1], 30, ' dos clústeres', va = 'center', fontdict={'size': 15})
ax.text(limites[1], 26, ' tres clústeres', va = 'center', fontdict={'size': 15})
ax.set_xlabel("Clientes")
ax.set_ylabel("Distancia o Agregación")
plt.show()

In [ ]:
grupos_h = fcluster(linkage(datos_std, method = 'ward', metric='euclidean'), 3, criterion = 'maxclust')
grupos_h = grupos_h-1 # Se resta 1 para que los clústeres se enumeren de 0 a (K-1), como usualmente lo hace Python
pd.Series(grupos_h).value_counts().sort_index()

In [ ]:
centros_h = np.array(pd.concat([ns.centroide(0, datos_std, grupos_h),
                                ns.centroide(1, datos_std, grupos_h),
                                ns.centroide(2, datos_std, grupos_h)]))
ns.radar_plot(centros_h, datos_std.columns)
plt.show()

In [ ]:
datos["Cluster_H"] = grupos_h
datos.groupby("Cluster_H")[datos_numericos.columns].mean().round(2)

### Comparación K-medias vs Jerárquico (Ward)

In [ ]:
pd.crosstab(datos["Cluster_KM"], datos["Cluster_H"], rownames=["K-medias"], colnames=["Ward"])

**Ward con 3 clústeres:**
- **Cluster 0 (6 clientes):** grandes compradores con gasto muy alto en todas las categorías, sobre todo `Grocery`, `Milk` y `Detergents_Paper`.
- **Cluster 1 (153 clientes):** perfil *Retail* (abarrotes, lácteos y detergentes); 113 son Retail.
- **Cluster 2 (281 clientes):** perfil *Horeca* (frescos y congelados); 257 son Horeca.

**Comparación:** ambos métodos coinciden en la segmentación principal: el Cluster 1 de K-medias (Horeca) coincide en su mayoría con el Cluster 2 de Ward (272 de 315), y el Cluster 0 de K-medias (Retail) cae casi por completo en el Cluster 1 de Ward (105 de 110). La diferencia principal está en los clientes atípicos: K-medias los separa en dos grupos (grandes minoristas y grandes compradores de frescos), mientras que Ward los mezcla en uno solo o los asigna a los grupos grandes. Además, Ward pone más clientes en el grupo Retail (153 vs 110).

## 5. Conclusiones
1. El set de datos tiene 440 clientes, sin valores nulos ni duplicados, con 6 variables numéricas de gasto y 2 variables categóricas (`Channel`, `Region`) que se usaron solo para interpretar.
2. Las variables de gasto son muy asimétricas, tienen escalas distintas y muchos atípicos; por eso se **estandarizaron** antes de aplicar métodos basados en distancia euclidiana.
3. El ACP y la matriz de correlación muestran dos patrones de compra: uno de **abarrotes, lácteos y detergentes** (Retail) y otro de **frescos, congelados y delicatessen** (Horeca).
4. K-medias con **K = 4** encuentra dos segmentos principales (Horeca típicos y Retail típicos) y dos segmentos pequeños de clientes de gran volumen, que el negocio podría atender de forma diferenciada (por ejemplo, con precios por volumen o atención personalizada).
5. El método jerárquico de Ward confirma la segmentación principal Horeca/Retail, lo que da confianza en los resultados.
6. La región no resultó relevante para diferenciar a los clientes; el patrón de compra está mucho más ligado al **canal**.

In [ ]:
datos.to_csv("Wholesale_clusters.csv", index = False)